# Task 16 — Kiểm định Thống kê (IS403), Trực quan hóa Ma trận & Đóng gói Cuối kỳ

- **Môn học:** Phân tích Dữ liệu Kinh doanh (`IS403` — UIT, ĐHQG-HCM)
- **Phạm vi tổng hợp:** Trọn bộ **Task 1 → Task 16** trên tập dữ liệu `FreshRetailNet-50K` (`50.000` chuỗi SKU–cửa hàng, `4.500.000` dòng `train`, `350.000` dòng `eval`, `206.562` quan sát không hết hàng hợp lệ).
- **Nội dung trọng tâm:**
  1. Đối soát truy xuất nguồn gốc SHA-256 và toàn bộ chỉ số Bảng 2 (`TimesNet`, `DLinear`) & Bảng 3 (`8` cấu hình trên `SSA`, `DLinear`, `TFT`).
  2. Vận dụng hệ thống kiểm định thống kê **Chương 2 môn IS403** (*Paired Student's $t$-test*, *Wilcoxon Signed-Rank*, *Tiền kiểm định Levene*, *One-Way ANOVA*, *Hậu kiểm Tukey's HSD*, *Friedman Block Test*, *Two-Way ANOVA* và *Kiểm định tương đương tái lập*).
  3. Chứng minh bằng số liệu thực chứng trên `4.500.000` dòng cơ chế gây thiên lệch dương ở `DLinear + RevIN` trên nhóm `Low Sale` (`50.43%` chuỗi có $\sigma < 0.30$) đối lập với tính ổn định của `SSA` và `TFT`.
  4. Xuất bộ **6 biểu đồ 300 DPI**, quyển **Báo cáo tổng kết `.docx`** và **Slide thuyết trình cuối kỳ**.

## 1. Kiểm định Truy xuất Nguồn gốc Dữ liệu & Hiện vật Thực nghiệm

In [1]:
import json, pathlib
import numpy as np
import pandas as pd
from scipy import stats

CWD = pathlib.Path.cwd().resolve()
ROOT = next((p for p in [CWD, *CWD.parents] if (p / "Task16_Final_Synthesis").is_dir()), CWD)
RES_DIR = ROOT / "Task16_Final_Synthesis" / "results"
FIG_DIR = RES_DIR / "figures"

df_art   = pd.read_csv(RES_DIR / "master_artifacts_index.csv", keep_default_na=False)
df_t2    = pd.read_csv(RES_DIR / "table2_recovery_summary.csv", keep_default_na=False)
df_t3    = pd.read_csv(RES_DIR / "table3_all_8configs_summary.csv", keep_default_na=False)
df_reg   = pd.read_csv(RES_DIR / "recovery_regime_mechanism.csv", keep_default_na=False)
df_daily = pd.read_csv(RES_DIR / "ssa_daily_7days_metrics.csv", keep_default_na=False)
df_stats = pd.read_csv(RES_DIR / "statistical_tests_is403.csv", keep_default_na=False)

print("=== DANH MỤC DỮ LIỆU PARQUET & CHECKPOINT ĐÃ XÁC MINH SHA-256 ===")
print(df_art[["artifact_name", "source_task", "size_mib", "verified_status"]].to_string(index=False))

=== DANH MỤC DỮ LIỆU PARQUET & CHECKPOINT ĐÃ XÁC MINH SHA-256 ===
                 artifact_name                          source_task size_mib        verified_status
                 train.parquet    Task 6 (Có sẵn trong GitHub repo)    58.25        sha256_verified
                  eval.parquet    Task 6 (Có sẵn trong GitHub repo)     5.20        sha256_verified
       demand_timesnet.parquet Task 10 — Minh Nguyễn (Google Drive)    77.78        sha256_verified
        demand_dlinear.parquet       Task 12 — Khanh (Google Drive)    77.75        sha256_verified
TimesNet_Recovery_Drive_Folder                Task 10 — Minh Nguyễn          documented_public_link
 DLinear_Recovery_Drive_Folder                      Task 12 — Khanh          documented_public_link
TFT_Checkpoints_lightning_logs                Task 15 — Minh Nguyễn          documented_public_link


## 2. Tổng hợp Kết quả Tự chạy Bảng 2 (Giai đoạn 1) và Bảng 3 (Giai đoạn 2 — 8 Cấu hình)

In [2]:
print("=== BẢNG 2: KHÔI PHỤC NHU CẦU TIỀM ẨN (TIMESNET VS DLINEAR) ===")
print(df_t2[["model", "source_task", "mnar_wape_pct", "mnar_wpe_pct", "decoupling_score_4dp", "paper_mnar_wape_pct", "paper_decoupling_score"]].to_string(index=False))

print("\n=== BẢNG 3: TRỌN BỘ 8 CẤU HÌNH DỰ BÁO 7 NGÀY (24 DÒNG — 206.562 MẪU HỢP LỆ) ===")
print(df_t3.to_string(index=False))

=== BẢNG 2: KHÔI PHỤC NHU CẦU TIỀM ẨN (TIMESNET VS DLINEAR) ===
    model       source_task mnar_wape_pct mnar_wpe_pct  decoupling_score_4dp paper_mnar_wape_pct  paper_decoupling_score
Raw Sales Task 10 & Task 12                                          -0.5664                                       -0.57
  DLinear           Task 12         30.32       -14.38               -0.1512               29.99                   -0.16
 TimesNet  Task 9 & Task 10         27.81         2.04                0.0699               27.62                    0.07

=== BẢNG 3: TRỌN BỘ 8 CẤU HÌNH DỰ BÁO 7 NGÀY (24 DÒNG — 206.562 MẪU HỢP LỆ) ===
       config_id     group  wape_pct  wpe_pct    mae paper_wape_pct paper_wpe_pct delta_wape_pp delta_wpe_pp    status
    SSA_RawSales   Overall     31.97   -10.50 0.3982          31.97        -10.50          0.00         0.00 completed
     SSA_DLinear   Overall     30.12    -0.58 0.3749          30.00         -1.49          0.12         0.91 completed
    SSA_TimesN

## 3. Kiểm định Giả thuyết Thống kê Chương 2 IS403 (`Paired t`, `Wilcoxon`, `Levene`, `ANOVA`, `Tukey HSD`, `Friedman`)

In [3]:
print("=== KẾT QUẢ KIỂM ĐỊNH GIẢ THUYẾT THỐNG KÊ CHƯƠNG 2 IS403 (PAIRED T-TEST, WILCOXON, LEVENE, ANOVA, TUKEY HSD, FRIEDMAN) ===")
cols_show = ["test_id", "scope_group", "sample_unit_n", "mean_or_effect", "primary_stat", "primary_p_value", "decision_alpha_0_05"]
print(df_stats[cols_show].to_string(index=False))

=== KẾT QUẢ KIỂM ĐỊNH GIẢ THUYẾT THỐNG KÊ CHƯƠNG 2 IS403 (PAIRED T-TEST, WILCOXON, LEVENE, ANOVA, TUKEY HSD, FRIEDMAN) ===
                             test_id                    scope_group  sample_unit_n                                          mean_or_effect                           primary_stat            primary_p_value                                        decision_alpha_0_05
  PAIRED_SKU_OVERALL_ae_raw_vs_ae_tn                        Overall          50000                                               -0.035668                           t = -31.5897                   < 1e-100                  Bác bỏ H0 (Khác biệt có ý nghĩa thống kê)
  PAIRED_SKU_OVERALL_ae_raw_vs_ae_dl                        Overall          50000                                               -0.030732                           t = -36.9431                   < 1e-100                  Bác bỏ H0 (Khác biệt có ý nghĩa thống kê)
   PAIRED_SKU_OVERALL_ae_dl_vs_ae_tn                        Overall          50000   

## 4. Giải mã Cơ chế Nghiệp vụ (`SSA` vs `DLinear + RevIN` vs `TFT`) & Kiểm tra Bộ 6 Biểu đồ

In [4]:
print("=== PHÂN TÍCH ĐỊNH LƯỢNG CƠ CHẾ NGHIỆP VỤ TRÊN 4.500.000 DÒNG TRAIN (LOW SALE VS HIGH SALE) ===")
print(df_reg.to_string(index=False))

print("\n=== KIỂM TRA 6 BIỂU ĐỒ TỔNG HỢP CUỐI KỲ (300 DPI) ===")
for p in sorted(FIG_DIR.glob("*.png")):
    print(f"  [OK] {p.name:48s} | {p.stat().st_size / 1024:6.1f} KiB")

=== PHÂN TÍCH ĐỊNH LƯỢNG CƠ CHẾ NGHIỆP VỤ TRÊN 4.500.000 DÒNG TRAIN (LOW SALE VS HIGH SALE) ===
    group  n_series  stockout_day_rate_pct  stockout_hour_rate_pct  zero_raw_sale_day_pct  raw_daily_mean  dlinear_rec_daily_mean  timesnet_rec_daily_mean  dlinear_uplift_pct  timesnet_uplift_pct  seq62_std_raw  seq62_std_dlinear  seq62_std_timesnet  series_std_lt_0_30_pct
  Overall     50000                  44.27                   19.88                   4.46          0.9986                  1.1438                   1.2065               14.54                20.82         0.5060             0.5065              0.5032                   36.90
 Low Sale     36449                  42.81                   19.73                   5.05          0.5979                  0.6966                   0.7197               16.51                20.38         0.3398             0.3555              0.3554                   50.43
High Sale     13551                  48.19                   20.28                